## Automated Description Generation
Here we can use the Python SDK to develop an automated description generation workflow, then save it to a config.yaml and run it from there.

This workflow generates descriptions for Milvus collections by analyzing a subset of their contents.

**Prerequisites:**
- Milvus server running at `localhost:19530`
- A collection named `wikipedia_docs` with embedded documents


In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)


In [ ]:
from nat.agent.react_agent.register import NatReActAgent
from nat.embedder.nim_embedder import NIMEmbedder
from nat.llm.nim_llm import NimLLM
from nat.retriever.milvus.register import MilvusRetriever
from nat.tool.retriever import NatRetrieverTool
from nat.utils.sdk.nat_workflow import NatWorkflow
from nat_automated_description_generation.register import AutomatedDescriptionMilvusWorkflow

# Create the LLM
llm = NimLLM(
    model_name="nvdev/meta/llama-3.1-70b-instruct",
    temperature=0.0,
    max_tokens=10000,
    name="nim_llm",
)

# Create the embedder
embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    truncate="END",
    name="milvus_embedder",
)

# Create the Milvus retriever
retriever = MilvusRetriever(
    uri="http://localhost:19530",
    collection_name="wikipedia_docs",
    embedder=embedder,
    top_k=10,
    name="retriever",
)

# Create the base retrieval tool
cuda_tool = NatRetrieverTool(
    nat_retriever=retriever,
    topic="NVIDIA CUDA",
    description="This tool can only retrieve information about NVIDIA's CUDA library.",
    name="cuda_tool",
)

# Create the automated description workflow
retrieve_tool = AutomatedDescriptionMilvusWorkflow(
    llm=llm,
    retriever=retriever,
    retrieval_tool=cuda_tool,
    collection_name="wikipedia_docs",
    name="retrieve_tool",
)

# Create the agent with the automated description tool
agent = NatReActAgent(
    tools=[retrieve_tool],
    llm=llm,
    verbose=True,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)


In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())


In [ ]:
# Query the collection with automatically generated description
await nat_workflow.prompt("What is CUDA?")
